# Lab Exploration 2 - LM-Eval-Harness

For this lab, you will explore using the Language Model Evaluation Harness (LM-Eval) from EleutherAI. To start, you may be interested in looking over it's associated paper found [here](https://arxiv.org/pdf/2405.14782). LM-Eval is open source library for running large language models through evaluation.

LM-Eval has a complex infrastructure, and a significant part of this lab is about gaining experience interacting with complicated open-source or research-focused packages. This includes navigation and troubleshooting package issues. This skill will probably be helpful for your final project, and it's also great applied experience.

Navigate to the [lm-eval repository](https://github.com/EleutherAI/lm-evaluation-harness). Look through some of it's documentation and become familiar with what using the package looks like.

Your first task is to get lm-eval setup. You'll notice that the install instructions for lm-eval require you to clone the repository and install with pip. You can do this in Colab Pro by pulling up the terminal: select terminal at the bottom of the page. If you're on the free version, you can also just run the following commands:

In [1]:
%%shell
git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness
cd lm-evaluation-harness
pip install -e .

Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 16050, done.
remote: Counting objects: 100% (16050/16050), done.
remote: Compressing objects: 100% (6349/6349), done.
remote: Total 16050 (delta 9754), reused 15790 (delta 9671), pack-reused 0 (from 0)
Receiving objects: 100% (16050/16050), 4.41 MiB | 10.28 MiB/s, done.
Resolving deltas: 100% (9754/9754), done.
Obtaining file:///content/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 12.4 MB/s eta 0:00:00
 

In [1]:
# Install both hf and vllm backends
!pip install "lm_eval[hf,vllm]" jedi protobuf==5.29.5

In [2]:
import sys
sys.path.append('/content/lm-evaluation-harness') # Add package directory to path, necessary in Colab
import lm_eval
import json
from lm_eval.utils import handle_non_serializable
from huggingface_hub import login
from lm_eval.utils import setup_logging
# Set log level
setup_logging("DEBUG")  # DEBUG, INFO, WARNING, ERROR

Login to HuggingFace Hub; some datasets and models are gated, meaning you will have to agree to Terms and Conditions before you can use them.

In [3]:
login()

To use LM-Eval, you can directly run it through the shell:

In [ ]:
%%shell
lm_eval --model hf \
    --model_args pretrained=openai-community/gpt2 \
    --tasks hellaswag \
    --device cuda:0 \
    --batch_size 16

2026-02-02:14:40:07 INFO     [_cli.run:376] Selected Tasks: ['hellaswag']
2026-02-02:14:40:08 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-02:14:40:08 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'openai-community/gpt2'}
2026-02-02:14:40:11 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-02:14:40:12 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-02 14:40:12.881952: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 14:40:12.898807: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to regist

Alternatively, you can also use the Python API:

In [ ]:
models = ["gpt2"]

all_results = {}
for model_name in models:
    results = lm_eval.simple_evaluate(
        model="hf",
        model_args=f"pretrained={model_name}",
        tasks=["hellaswag"],
        batch_size="auto",
    )
    all_results[model_name] = results["results"]

print(all_results)

# Save results
with open("results.json", "w") as f:
    json.dump(results, f, default=handle_non_serializable, indent=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Running loglikelihood requests:   0%|          | 0/40168 [00:00<?, ?it/s]

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 64


Running loglikelihood requests: 100%|██████████| 40168/40168 [01:05<00:00, 610.80it/s] 


{'gpt2': {'hellaswag': {'alias': 'hellaswag', 'acc,none': 0.2891854212308305, 'acc_stderr,none': 0.004524575892953094, 'acc_norm,none': 0.31139215295757816, 'acc_norm_stderr,none': 0.004621163476949437}}}


For this lab, your tasks are to:
- Read through python api and tasks: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/python-api.md
- Pick at least 3 tasks that sound interesting to you, and learn about each of them.
- Pick 2 models to compare on each of these tasks.
- Attempt to run both models across all tasks, and plot/table results.
- If you run into issues running one of the tasks, dig into the problem a bit and see if you can fix it, or at least understand what is happening (we're expecting most of your time will end up being here).
- Compare the scores to the high scores for each benchmark.

Some tips:
- It might be a good idea to run all of the benchmarks you want to try on a very small model first, to see if they work, before scaling up to a large model.
- Some good small models to test are HuggingFaceTB/SmolLM-360M or meta-llama/Llama-3.2-1B-Instruct. Any larger than 1B and you might start running into memory issues.
- Be aware, some benchmarks are very large and may take a very long time to finish (like, 10+ hours in Colab). For example, bbh. If this is the case, you can try increasing the batch size, or just try a different benchmark.

Other directions you may look into:
- In class, we are reading the MMLU paper. In LM-Eval, you have access to several versions of mmlu, including mmlu_pro, mmlu-pro-plus, mmlu_prox, and mmlusr. Try to run a model across all of these, and compare the differences in scores. What differences do you notice?
- Try using the vllm backend instead of hf, if you haven't yet. It /should/ be faster. (I personally couldn't get it to work)
- Look into what it takes to incorporate a new benchmark into lm-eval.

A final tip: you can track your GPU memory usage by hitting the RAM/Disk usage graphs in the upper right-hand corner. Sometimes, your GPU memory might not be properly released when running these benchmarks. To fix this, you can either restart the kernel (Runtime->Restart session) or try to run the following code:

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# experimentation here

When you are finished, upload to LearningSuite:

1. Your Python notebook
2. A PDF document containing answers to the following questions:

- What did you do? Why did you choose to do this?
- What did you learn from this experimentation?
- Why do you think this is interesting? Or, how can you apply this to the class or your own research?
- Did you spend adequate (~5 hours) of time on this?

Feel free to attach any images outlining your results to your report.